In [1]:
import pandas as pd
import numpy as np

In [2]:
# LOAD DATA
df = pd.read_excel("Input_TTF_NG_Real_End_of_Month_Prices.xlsx")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df[["date", "price_real"]]
#price_real = Real_Monthly_Average_TTF_NG_Prices
#date = the last day of the month for which the monthly average price is calcualted

In [3]:
df

,date,price_real
0,2006-02-28,25.830138
1,2006-03-31,26.049717
2,2006-04-30,23.752324
3,2006-05-31,23.272535
4,2006-06-30,20.797866
...,...,...
234,2025-08-31,23.996138
235,2025-09-30,24.693700
236,2025-10-31,23.400748
237,2025-11-30,22.119086


In [4]:
#RANDOM WALK FORECASTS (no drift)
horizons = [1, 3, 6, 9, 12, 15, 18, 21, 24]
# Store results: each row = one forecast origin
results = []

for i in range(len(df)):
    origin_date = df.loc[i, "date"]
    origin_price = df.loc[i, "price_real"]

    row = {"forecast_origin": origin_date}

    for h in horizons:
        # RW forecast = current price, regardless of horizon
        row[f"RW_h{h}"] = origin_price
        
#Actual Prices calculation:
#Don't include this part when using END_OF_MONTH data, so that it doesn't calcualte actual prices using the END_OF_MONTH data!
        # Also store actual price at t+h for MSPE calculation later
        target_idx = i + h
        if target_idx < len(df):
            row[f"actual_h{h}"] = df.loc[target_idx, "price_real"]
        else:
            row[f"actual_h{h}"] = np.nan

    results.append(row)

df_rw = pd.DataFrame(results)

In [5]:
# COMPUTE MSPE FOR EACH HORIZON
print("Random Walk MSPE by horizon:\n")
mspe_rw = {}

for h in horizons:
    errors = df_rw[f"actual_h{h}"] - df_rw[f"RW_h{h}"]
    mspe = (errors ** 2).mean()
    mspe_rw[h] = mspe
    print(f"  h={h:2d}: MSPE = {mspe:.4f}")

Random Walk MSPE by horizon:

  h= 1: MSPE = 178.4634
  h= 3: MSPE = 326.0476
  h= 6: MSPE = 516.0085
  h= 9: MSPE = 714.7013
  h=12: MSPE = 903.9546
  h=15: MSPE = 1085.9614
  h=18: MSPE = 1220.8294
  h=21: MSPE = 1262.5221
  h=24: MSPE = 1284.2314


In [6]:
# Convert wide RW forecasts to long format
records = []

for _, row in df_rw.iterrows():
    for h in horizons:
        records.append({
            "forecast_origin": row["forecast_origin"],
            "horizon": h,
            "model": "RW",
            "forecast": row[f"RW_h{h}"],
            "actual": row[f"actual_h{h}"]
        })

df_rw_long = pd.DataFrame(records)
df_rw_long = df_rw_long.dropna(subset=["actual"])  # drop where actual not yet available

df_rw_long.to_excel("rw_forecasts_long.xlsx", index=False)
print(df_rw_long.head(20))

   forecast_origin  horizon model   forecast     actual
0       2006-02-28        1    RW  25.830138  26.049717
1       2006-02-28        3    RW  25.830138  23.272535
2       2006-02-28        6    RW  25.830138  19.370563
3       2006-02-28        9    RW  25.830138  20.255334
4       2006-02-28       12    RW  25.830138  11.027823
5       2006-02-28       15    RW  25.830138  12.379240
6       2006-02-28       18    RW  25.830138  16.129705
7       2006-02-28       21    RW  25.830138  22.661509
8       2006-02-28       24    RW  25.830138  26.265305
9       2006-03-31        1    RW  26.049717  23.752324
10      2006-03-31        3    RW  26.049717  20.797866
11      2006-03-31        6    RW  26.049717  16.194079
12      2006-03-31        9    RW  26.049717  16.060988
13      2006-03-31       12    RW  26.049717  12.370665
14      2006-03-31       15    RW  26.049717  12.406303
15      2006-03-31       18    RW  26.049717  19.414914
16      2006-03-31       21    RW  26.049717  30

In [7]:
# SAVE FORECASTS
#df_rw.to_csv("rw_forecasts_long_format.csv", index=False)
#print("\nForecasts saved to rw_forecasts.csv")